In [ ]:
import torch
import torch.nn as nn
from transformers import AutoImageProcessor, AutoModel

import requests
from PIL import Image, ImageOps
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import os
import cv2

In [ ]:
import random

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
MODEL_PATH = "../models/dinov2b.pth"
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
HF_MODEL_NAME = "facebook/dinov2-base"
EMBEDDING_DIM  = 256
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
def load_image(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"Error loading {url}: {e}")
        return None

def pad_to_square(img, background_color=(255, 255, 255)):
    width, height = img.size
    if width == height:
        return img
    elif width > height:
        padding = (0, (width - height) // 2, 0, (width - height + 1) // 2)
    else:
        padding = ((height - width) // 2, 0, (height - width + 1) // 2, 0)
    return ImageOps.expand(img, padding, fill=background_color)

def gray_image(img):
    img = np.array(img)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray_img = cv2.merge([gray, gray, gray])
    gray_img = Image.fromarray(gray_img)
    return gray_img

def load_images_parallel(urls, max_workers=32):
    results = [None] * len(urls)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(load_image, url): i for i, url in enumerate(urls)}
        with tqdm(total=len(futures), desc="Downloading images", position=0, leave=False) as pbar:
            for future in as_completed(futures):
                i = futures[future]
                try:
                    results[i] = future.result()
                except Exception as e:
                    print(f"Error in future: {e}")
                pbar.update(1)
    return results

def chunked_iterable(iterable, size):
    for i in range(0, len(iterable), size):
        yield iterable[i:i + size]

def preprocess_image(image):
    img = image.convert("RGB")
    padded_img = pad_to_square(img, background_color=(255, 255, 255))
    gray_img = gray_image(padded_img)
    return gray_img

def img2vec(image, processor, model):
    model.eval()
    inputs = processor(image, return_tensors = "pt")
    pixel_values = inputs["pixel_values"].to(device)
    with torch.no_grad():
        embedding = model(pixel_values)
    return embedding.squeeze(0).cpu().numpy().tolist()

In [ ]:
def process(images, processor, model):
    vectors = []
    for img in tqdm(images, desc="Processing images"):
        if img is None:
            vectors.append(None)
            continue
        try:
            img = preprocess_image(img)
            vec = img2vec(img, processor, model)
            vectors.append(vec)
        except Exception as e:
            print(f"Processing error: {e}")
            vectors.append(None)
    return vectors

def url2vec(url, processor, model):
    try:
        img=load_image(url)
        gray_img=preprocess_image(img)
        vec=img2vec(gray_img, processor, model)
        return vec
    except requests.exceptions.RequestException as e:
        #print(f"Failed to fetch image from URL: {url}\nError: {e}")
        return None
    except Exception as e:
        #print(f"Error processing image from URL: {url}\nError: {e}")
        return None

In [ ]:
class DinoV2HFWithProjection(nn.Module):
    def __init__(self, model_name_hf=HF_MODEL_NAME, embedding_dim=EMBEDDING_DIM):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name_hf)
        self.backbone_feature_dim = self.backbone.config.hidden_size
        self.projection_head = nn.Linear(self.backbone_feature_dim, embedding_dim)

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        features = outputs.last_hidden_state[:, 0]
        embeddings = self.projection_head(features)
        return embeddings

In [ ]:
model = DinoV2HFWithProjection()
try:
    checkpoint = torch.load(MODEL_PATH)
    model.load_state_dict(checkpoint, strict=True)  # Add strict=True
    print("Model loaded successfully")
except Exception as e:
    print(f"Error loading model: {e}")
model.eval().to(device)

In [ ]:
_ = model(torch.rand(1, 3, 224, 224).to(device))

# Vectorize

In [ ]:
df=pd.read_csv("data_clean.csv")
mask = df[df["vectors"].isna()]
urls=mask["ImageURL"].to_list()

In [ ]:
vectors = []

batch_size = 256
for batch_urls in tqdm(list(chunked_iterable(urls, batch_size)), desc="Processing batches"):
    images = load_images_parallel(batch_urls, max_workers=8)
    vector = process(images, processor, model)
    vectors.extend(vector)
    del images
    del vector
    torch.cuda.empty_cache()

In [ ]:
vectors[0]

In [ ]:
print(f"{type(vectors[0])} {type(vectors)} {len(vectors[0])} {len(vectors)}")

In [ ]:
mask["vectors"] = vectors

In [ ]:
mask.info()

In [ ]:
df.update(mask)

In [ ]:
df.info()

In [ ]:
df.to_csv("vectors.csv", index=False)

# Test

In [ ]:
url = 'https://www.tjc.co.uk/dw/image/v2/AAXN_PRD/on/demandware.static/-/Sites-master-catalog-tjc/default/dwdea9d73a/Large/21/5/2153532.jpg'
vector =  url2vec(url, processor, model)

In [ ]:
print(f"vector:{vector}, length:{len(vector)}, datatpe:{type(vector)}")